# Agent Evaluation with MLflow Prompt Registry

This notebook demonstrates how to evaluate the Customer Analyst agent using MLflow's evaluation framework with **Prompt Registry integration**.

Features:
- **Prompt Registry**: Register the agent's system prompt as a versioned artifact in MLflow
- **Trace-to-Prompt Linkage**: Automatically link evaluation traces to the prompt version that produced them
- **Dataset Creation**: Create persistent evaluation datasets on the MLflow server (from test cases and annotated traces)
- **Simple mode**: Fast, deterministic checks (no LLM calls)
- **LLM-as-a-Judge mode**: Full evaluation with LLM judges for tool correctness, safety, and relevance

## Prerequisites

1. Install the Red Hat MLflow fork (cell below)
2. Set the environment variables (cell below)
3. Get an MLflow tracking token:
   ```bash
   export MLFLOW_TRACKING_TOKEN=$(oc whoami --show-token)
   ```

## Install Dependencies

In [ ]:
%pip install \
  "a2a-sdk[http-server]>=0.3.25" \
  "httpx>=0.27.0" \
  "pydantic>=2.0.0" \
  "uvicorn>=0.27.0" \
  "fastmcp>=3.0.0,<4" \
  "mlflow" \
  "nest-asyncio" \
  "langchain"

import os

import nest_asyncio
nest_asyncio.apply()

# -- Set your environment variables here --
os.environ["LANG_MODEL_ENDPOINT"] = "https://litellm-prod.apps.maas.redhatworkshops.io/v1"
os.environ["LANG_MODEL_API_KEY"] = ""
os.environ["LANG_MODEL_NAME"] = "llama-scout-17b"
os.environ["MLFLOW_TRACKING_URI"] = "http://127.0.0.1:5000"
os.environ["MLFLOW_EXPERIMENT_NAME"] = "test-dev-2"
os.environ["EVENT_HUB_URL"] = ""
#os.environ["MLFLOW_TRACKING_AUTH"] = "kubernetes"
#os.environ["MLFLOW_TRACKING_INSECURE_TLS"] = "true"
# os.environ["MLFLOW_TRACKING_TOKEN"] = ""  # or run: export MLFLOW_TRACKING_TOKEN=$(oc whoami --show-token)

## Setup

In [ ]:
# This project was developed with assistance from AI tools.
import sys
import traceback
import warnings
from pathlib import Path

# Suppress warnings
warnings.filterwarnings("ignore")

repo_root = Path.cwd().parent
#sys.path.insert(0, str(repo_root))
sys.path.insert(0, str(Path.cwd().parent / "services" / "customer-analyst"))
sys.path.insert(0, str(Path.cwd().parent / "services" / "mongodb-mcp"))
print(sys.path)

# Verify required environment variables
required_vars = [
    "MLFLOW_TRACKING_URI",
    "MLFLOW_EXPERIMENT_NAME",
    "LANG_MODEL_ENDPOINT",
    "LANG_MODEL_NAME",
]
missing = [v for v in required_vars if not os.environ.get(v)]
if missing:
    raise ValueError(f"Missing required environment variables: {', '.join(missing)}")

print(f"MLflow URI: {os.environ.get('MLFLOW_TRACKING_URI')}")
print(f"Experiment: {os.environ.get('MLFLOW_EXPERIMENT_NAME')}")
print(f"LLM Base URL: {os.environ.get('LANG_MODEL_ENDPOINT')}")
print(f"LLM Model: {os.environ.get('LANG_MODEL_NAME')}")
#print(f"Tracking Auth: {os.environ.get('MLFLOW_TRACKING_AUTH')}")

## Configure MLflow

In [ ]:
import logging
from pathlib import Path
import mlflow

# Suppress MLflow warnings
logging.getLogger("mlflow").setLevel(logging.ERROR)

# Auto-detect Kubernetes service account token if not set
if not os.environ.get("MLFLOW_TRACKING_TOKEN"):
    sa_token_path = Path("/var/run/secrets/kubernetes.io/serviceaccount/token")
    if sa_token_path.exists():
        os.environ["MLFLOW_TRACKING_TOKEN"] = sa_token_path.read_text().strip()
        print("Auto-detected Kubernetes SA token")
    else:
        print("WARNING: MLFLOW_TRACKING_TOKEN not set and no SA token found.")
        print("Run: export MLFLOW_TRACKING_TOKEN=$(oc whoami --show-token)")

# Configure MLflow
mlflow.set_tracking_uri(os.environ["MLFLOW_TRACKING_URI"])

experiment_name = os.environ["MLFLOW_EXPERIMENT_NAME"]

mlflow.set_experiment(experiment_name)
experiment = mlflow.get_experiment_by_name(experiment_name)

mlflow.langchain.autolog()

print(f"Experiment set: {experiment_name}")
print(f"Experiment ID: {experiment.experiment_id}")

## 1. Register System Prompt in MLflow

Register the public-assistant system prompt in MLflow's Prompt Registry as a versioned artifact. The prompt text below is the same content from `config/agents/public-assistant.yaml`.

In [ ]:
system_prompt_text = """You are a customer data analyst for a luxury casino resort. Given a target audience description, decide which database tool to call to retrieve the right customer segment. You have access to the following tools:

- get_customers_by_tier: For specific tier queries (platinum, gold, diamond members)
- get_prospects: For new/potential customers who aren't members yet
- get_high_spend_customers: For high-spending VIP/whale customers
- get_all_vip_customers: For broad targeting across all VIP tiers

Call exactly ONE tool based on the target audience description. Do not call multiple tools."""

print(f"Prompt length: {len(system_prompt_text)} characters")
print(f"\nFirst 200 chars:\n{system_prompt_text[:200]}...")

In [ ]:
# Register the prompt in MLflow Prompt Registry
prompt_name = "customer-analyst-system-prompt"

try:
    registered_prompt = mlflow.genai.register_prompt(
        name=prompt_name,
        template=system_prompt_text,
        commit_message="Customer Analyst system prompt from config/agents/public-assistant.yaml",
        tags={
            "agent": "customer-analyst",
            "type": "system-prompt",
            "persona": "prospect",
            "source": "config/agents/public-assistant.yaml",
        },
    )
    print(f"Registered prompt: {prompt_name} (version {registered_prompt.version})")
except Exception as e:
    if "already exists" in str(e).lower() or "RESOURCE_ALREADY_EXISTS" in str(e):
        # Prompt already registered -- create a new version
        registered_prompt = mlflow.genai.register_prompt(
            name=prompt_name,
            template=system_prompt_text,
            commit_message="Updated Customer Analyst system prompt",
            tags={
                "agent": "customer-analyst",
                "type": "system-prompt",
                "persona": "prospect",
                "source": "config/agents/public-assistant.yaml",
            },
        )
        print(f"Updated prompt: {prompt_name} (version {registered_prompt.version})")
    else:
        traceback.print_exc()
        raise

# Store the prompt URI for trace linkage
prompt_uri = f"prompts:/{prompt_name}/{registered_prompt.version}"
print(f"Prompt URI: {prompt_uri}")

In [ ]:
# Verify by loading back from the registry
loaded_prompt = mlflow.genai.load_prompt(prompt_uri)
print(f"Loaded prompt: {prompt_name} (version {loaded_prompt.version})")
print(f"Template matches source: {loaded_prompt.template == system_prompt_text}")

## 2. Create Evaluation Dataset

Create a persistent dataset on the MLflow server. Each test case has:
- `inputs`: The user message to send to the agent
- `expectations`: Expected behavior including:
  - `expected_answer`: Keyword that should appear in the response
  - `expected_tool_calls`: Tools the agent should call (MLflow format)
  - `expected_topics`: Topics the response should cover
  - `forbidden_content`: Content that should NOT appear

In [ ]:
from mlflow.genai.datasets import create_dataset

test_cases = [
    {
        "inputs": {
            "user_message": "Retrieve customers for this target audience: platinum and identify the best prospects for the CNY VIP Gala campaign. Prioritize guests who enjoy exclusive dining, private salons, premium gaming, and luxury entertainment."
        },
        "expectations": {            
            "expected_tier": ["platinum"],
            "expected_tool_calls": [{"name": "get_customers_by_tier"}],
        },
    },
    {
        "inputs": {
            "user_message": "Retrieve customers for this target audience: platinum and find members most likely to respond to an ultra-exclusive Year of the Snake celebration with chef tasting menus, VIP hosts, and invitation-only access."
        },
        "expectations": {            
            "expected_tier": ["platinum"],
            "expected_tool_calls": [{"name": "get_customers_by_tier"}],
        },
    },
    {
        "inputs": {
            "user_message": "Retrieve customers for this target audience: gold and rank the best candidates for the Mid-Autumn Festival campaign. Focus on customers who appreciate festive experiences, mooncake tastings, and premium evening events."
        },
        "expectations": {            
            "expected_tier": ["gold"],
            "expected_tool_calls": [{"name": "get_customers_by_tier"}],
        },
    },
    {
        "inputs": {
            "user_message": "Retrieve customers for this target audience: gold and identify members who would enjoy a skyline lantern gala featuring cultural performances, elegant dining, and family-friendly celebrations."
        },
        "expectations": {    
            "expected_tier": ["gold"],
            "expected_tool_calls": [{"name": "get_customers_by_tier"}],
        },
    },
    {
        "inputs": {
            "user_message": "Retrieve all customers for this target audience: diamond and select the highest-value prospects for Diamond Weekend. Look for premium suite bookings, private transfers, and strong gaming spend."
        },
        "expectations": {            
            "expected_tier": ["diamond"],
            "expected_tool_calls": [{"name": "get_customers_by_tier"}],
        },
    },
    {
        "inputs": {
            "user_message": "Retrieve all customers for this target audience: diamond and identify patrons most likely to accept a private jet weekend escape with presidential suites, concierge services, and unlimited VIP access."
        },
        "expectations": {            
            "expected_tier": ["diamond"],
            "expected_tool_calls": [{"name": "get_customers_by_tier"}],
        },
    },
    {
        "inputs": {
            "user_message": "Retrieve customers for this target audience: new members and recommend the best candidates for the Welcome Experience campaign. Prioritize recently joined members with strong first-month engagement."
        },
        "expectations": {            
            "expected_tier": ["prospect"],
            "expected_tool_calls": [{"name": "get_prospects"}],
        },
    },
    {
        "inputs": {
            "user_message": "Retrieve customers for this target audience: new members and find guests likely to enjoy a VIP onboarding package with dining credits, spa vouchers, and a personal casino host."
        },
        "expectations": {            
            "expected_tier": ["prospect"],
            "expected_tool_calls": [{"name": "get_prospects"}],
        },
    },
    {
        "inputs": {
            "user_message": "Retrieve customers for this target audience: high-spend customers and identify the best audience for the Wellness Retreat campaign. Focus on spa usage, resort stays, and premium dining behavior."
        },
        "expectations": {            
            "expected_tier": ["platinum", "diamond", "gold"],
            "expected_tool_calls": [{"name": "get_high_spend_customers"}],
        },
    },
    {
        "inputs": {
            "user_message": "Retrieve customers for this target audience: high-spend customers and rank those most likely to book a luxury reset weekend with private yoga, daily spa treatments, and Michelin-starred dining."
        },
        "expectations": {
            "expected_tier": ["platinum", "diamond", "gold"],
            "expected_tool_calls": [{"name": "get_high_spend_customers"}],
        },
    },
]

# Create a new dataset on the MLflow server
dataset_name = "customer_analyst_eval"
dataset = create_dataset(
    name=dataset_name,
    tags={"stage": "validation", "version": "1", "agent": "customer-analyst"},
)

# Add test cases to the dataset
dataset = dataset.merge_records(test_cases)
print(f"Dataset created: {dataset.dataset_id}")
print(f"Test cases: {len(test_cases)}")

## 3. View the Dataset

Inspect the test cases we just created on the MLflow server.

In [ ]:
# View the dataset contents
print(f"Dataset: {dataset_name}")
print(f"Total test cases: {len(test_cases)}\n")

for i, case in enumerate(test_cases, 1):
    msg = case["inputs"]["user_message"]
    display_msg = msg[:60] + "..." if len(msg) > 60 else msg
    print(f"{i}. {display_msg}")
    print(f"   Expected tier: {case['expectations']['expected_tier']}")
    print(f"   Expected tools: {[t['name'] for t in case['expectations'].get('expected_tool_calls', [])]}")
    print()

## 4. Predictor with Prompt Linkage

The predictor wraps the agent invocation for MLflow's evaluate function. We enhance it to load the registered prompt inside the traced context, which causes MLflow to automatically link each evaluation trace to the prompt version.

In [ ]:
# import asyncio
# import contextvars
# import concurrent.futures

# from langchain_core.messages import HumanMessage
# from src.agents.registry import get_agent

# # Thread pool for running async code without contextvars conflicts
# _executor = concurrent.futures.ThreadPoolExecutor(max_workers=1)


# def _run_async(coro):
#     """Run an async coroutine in a separate thread with a copy of the current
#     context. This avoids Python 3.12 contextvars re-entry errors while
#     preserving MLflow/LangChain state needed for tracing and tool calling."""
#     ctx = contextvars.copy_context()
#     def _thread_target():
#         def _run():
#             loop = asyncio.new_event_loop()
#             try:
#                 return loop.run_until_complete(coro)
#             finally:
#                 loop.close()
#         return ctx.run(_run)
#     future = _executor.submit(_thread_target)
#     return future.result()


# def predict_fn_with_prompt(user_message: str) -> str:
#     """Predictor that invokes the agent and links traces to the registered prompt.

#     Loading the prompt inside a traced function creates automatic
#     trace-to-prompt linkage in MLflow.
#     """
#     if not user_message:
#         return "Error: No user_message provided"

#     async def _invoke() -> str:
#         try:
#             graph = get_agent("public-assistant", checkpointer=None)
#             initial_state = {
#                 "messages": [HumanMessage(content=user_message)],
#                 "user_role": "prospect",
#                 "user_id": "eval-user-001",
#                 "user_email": "evaluator@example.com",
#                 "user_name": "Evaluation User",
#                 "safety_blocked": False,
#                 "tool_allowed_roles": {},
#                 "decision_proposals": {},
#             }
#             result = await graph.ainvoke(initial_state)
#             messages = result.get("messages", [])
#             if messages:
#                 last_message = messages[-1]
#                 if hasattr(last_message, "content"):
#                     return str(last_message.content)
#                 return str(last_message)
#             return "Error: No response from agent"
#         except Exception as e:
#             return f"Error: {type(e).__name__}: {e}"

#     with mlflow.start_span(name="public-assistant-eval") as span:
#         span.set_inputs({"user_message": user_message})
#         # Load the prompt inside the trace to create prompt linkage
#         if prompt_uri:
#             mlflow.genai.load_prompt(prompt_uri)
#         result = _run_async(_invoke())
#         span.set_outputs({"response": result})
#         return result


# # Test it with a simple query
# print("Testing predictor with prompt linkage...")
# test_response = predict_fn_with_prompt("What loan products do you offer?")
# print(f"Prompt URI linked: {prompt_uri}")
# print(f"Test response preview: {test_response[:200]}...")

In [ ]:
# import importlib
# import agent

# importlib.reload(agent)

In [ ]:
import os
import agent #Not sure why jupyter notebook need to do this
from agent import CustomerAnalystAgent

# Mock MCP server
agent.call_mcp_tool = agent.fake_call_mcp_tool

if prompt_uri:
    mlflow.genai.load_prompt(prompt_uri)

agent = CustomerAnalystAgent()

@mlflow.trace
async def predict_fn_with_prompt(user_message: str):
        
    return await agent.get_customers(
        {
            "user_prompt": user_message,
            "campaign_id": "fake-campaign-id",
            "limit": 50,
        }
    )


# Test it with a simple query
print("Testing predictor with prompt linkage...")
test_response = await predict_fn_with_prompt("Get all platinum members")
print(f"Prompt URI linked: {prompt_uri}")
print(f"Test response preview: {test_response}...")

## 6. Scorers

We have two types of scorers:

### Simple Scorers (no LLM calls)
- `contains_expected`: Checks if expected keyword appears in response
- `has_numeric_result`: Checks if response contains numbers (for calculations)
- `response_length`: Ensures adequate response length

### LLM-as-a-Judge Scorers
- `ToolCallCorrectness`: Did the agent call the right tools?
- `ToolCallEfficiency`: Were tool calls minimal and efficient?
- `RelevanceToQuery`: Is the response relevant to the question?
- `Safety`: Is the response safe and appropriate?
- `Guidelines`: Does response follow mortgage assistant guidelines?

In [ ]:
from mlflow.genai.scorers import scorer

@scorer
def contain_expected_tier(inputs: dict, outputs: dict, expectations: dict) -> bool:
    """Check if customer tiers match expected tiers."""
    
    expected_tiers = expectations.get("expected_tier", [])
    customers = outputs.get("customers", [])

    #Al customer tiers must be within expected tiers
    return all(c.get("tier") in expected_tiers for c in customers)

@scorer
def has_at_least_one_customer(outputs: dict) -> bool:
    """Check must have 1 customer"""
    return outputs.get("count", 0) > 0
    
# Simple scorers (always available)
simple_scorers = [
    contain_expected_tier,
    has_at_least_one_customer,    
]

print("Simple scorers loaded:")
for s in simple_scorers:
    print(f"  - {s.name}")

In [ ]:
# # LLM-as-a-Judge scorers (requires LLM endpoint)
# from mlflow.genai.scorers import (
#     #Guidelines,
#     RelevanceToQuery,
#     #Safety,
#     ToolCallCorrectness,
#     #ToolCallEfficiency,
# )

# # Configure judge model
# base_url = os.environ.get("LANG_MODEL_ENDPOINT")
# model_name = os.environ.get("LANG_MODEL_NAME")
# api_key = os.environ.get("LANG_MODEL_API_KEY", "")

# if base_url and model_name:
#     os.environ["OPENAI_API_BASE"] = base_url
#     os.environ["OPENAI_BASE_URL"] = base_url
#     if api_key:
#        os.environ["OPENAI_API_KEY"] = api_key
#     judge_model = f"openai:/{model_name}"
#     print(f"Judge model: {judge_model}")

#     # Create LLM judge scorers
#     llm_scorers = [
#         ToolCallCorrectness(model=judge_model, should_exact_match=True),
#         #ToolCallEfficiency(model=judge_model),
#         #RelevanceToQuery(model=judge_model)          
#     ]
#     print(f"LLM judge scorers: {len(llm_scorers)}")
# else:
#     llm_scorers = []
#     print("LLM judge not configured - set LLM_BASE_URL and LLM_MODEL")

## 7. Run Simple Evaluation

Fast evaluation using only deterministic scorers. Traces are automatically linked to the registered prompt.

In [ ]:
# Run simple evaluation using the dataset
print("Running simple evaluation...")
print(f"Dataset: {len(test_cases)} examples")
print(f"Scorers: {len(simple_scorers)}")
print(f"Prompt: {prompt_uri}")
print()

with mlflow.start_run(run_name="eval-customer-analyst"):
    simple_results = mlflow.genai.evaluate(
        data=dataset,
        predict_fn=predict_fn_with_prompt,
        scorers=simple_scorers
    )

print("\nSimple Evaluation Results:")
print("-" * 40)
for metric, value in sorted(simple_results.metrics.items()):
    if isinstance(value, float):
        print(f"  {metric}: {value:.2%}")
    else:
        print(f"  {metric}: {value}")

## 8. Run Full LLM-as-a-Judge Evaluation

Complete evaluation including LLM judges for tool correctness, safety, and relevance.

In [ ]:
# if llm_scorers:
#     all_scorers = simple_scorers + llm_scorers

#     print("Running LLM-as-a-Judge evaluation...")
#     print(f"Dataset: {len(test_cases)} examples")
#     print(f"Scorers: {len(all_scorers)} ({len(llm_scorers)} LLM judges)")
#     print(f"Prompt: {prompt_uri}")
#     print()

#     full_results = mlflow.genai.evaluate(
#         data=dataset,
#         predict_fn=predict_fn_with_prompt,
#         scorers=all_scorers,
#     )

#     print("\nFull Evaluation Results:")
#     print("-" * 40)
#     for metric, value in sorted(full_results.metrics.items()):
#         if isinstance(value, float):
#             print(f"  {metric}: {value:.2%}")
#         else:
#             print(f"  {metric}: {value}")
# else:
#     print("Skipping LLM-as-a-Judge evaluation (not configured)")
#     print("Set LLM_BASE_URL and LLM_MODEL in your .env file")

## 9. View Results in MLflow

Navigate to the MLflow UI to see detailed results:

1. Open your MLflow tracking URI in a browser
2. Go to **Experiments** > your-experiment-eval
3. Click on **Evaluation Runs** tab
4. Select your run to view per-trace assessments
5. Enable **All Assessments** in the Columns dropdown to see LLM judge scores
6. Click on a trace to see the linked prompt version under **Prompt** tab

View registered prompts under **Prompts** in the left sidebar.
View datasets under **Experiments** > Default > **Datasets**.

In [ ]:
tracking_uri = os.environ.get("MLFLOW_TRACKING_URI", "")
print(f"View evaluation results: {tracking_uri}/#/experiments")
print(f"View registered prompts: {tracking_uri}/#/prompts")
print(f"Dataset ID: {dataset.dataset_id}")
print(f"Prompt: {prompt_uri}")